# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
# load_dotenv()
load_dotenv("config.env")

True

### VectorDB Instance

In [5]:
import logging

import chromadb
from chromadb.config import Settings

# Suppress non-critical ChromaDB telemetry log messages.
logging.getLogger(
    "chromadb.telemetry.product.posthog"
).setLevel(logging.CRITICAL)

# Disable anonymous telemetry for the ChromaDB client.
chroma_settings = Settings(
    anonymized_telemetry=False
)


In [6]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(
    path="chromadb",
    settings=chroma_settings
)

collection = chroma_client.get_collection("udaplay")

# print(type(chroma_client))
# print(chroma_client.heartbeat())

### Collection

In [7]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-ada-002"
)


In [8]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
    get_or_create=True
)

### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [10]:
print("Stored documents:", collection.count())

Stored documents: 15


In [11]:
stored_game = collection.get(ids=["001"])

print("ID:", stored_game["ids"][0])
print("Document:", stored_game["documents"][0])
print("Metadata:", stored_game["metadatas"][0])

ID: 001
Document: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
Metadata: {'Genre': 'Racing', 'Publisher': 'Sony Computer Entertainment', 'YearOfRelease': 1997, 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Platform': 'PlayStation 1', 'Name': 'Gran Turismo'}


In [12]:
query_results = collection.query(
    query_texts=["Which game is a realistic racing simulator?"],
    n_results=3
)

print("Query:", "Which game is a realistic racing simulator?")
print()

for rank, (document, metadata, distance) in enumerate(
    zip(
        query_results["documents"][0],
        query_results["metadatas"][0],
        query_results["distances"][0]
    ),
    start=1
):
    print(f"Result {rank}:")
    print("Game:", metadata["Name"])
    print("Document:", document)
    print("Distance:", distance)
    print()

Query: Which game is a realistic racing simulator?

Result 1:
Game: Gran Turismo
Document: [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
Distance: 0.2544863820075989

Result 2:
Game: Gran Turismo 5
Document: [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
Distance: 0.2800692915916443

Result 3:
Game: Mario Kart 8 Deluxe
Document: [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.
Distance: 0.4141138792037964

